# 01 EDA: DeepSolar Dataset Exploration

Understand data structure, target distribution, missing values, and prefix groups.

In [ ]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.features import load_data, get_prefix, get_us_only_cols, get_target_cols

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

## Load Data

In [ ]:
df = load_data('../deepsolar_tract.csv')
print('Shape:', df.shape)

## Target Variable: tile_count

In [ ]:
target = 'tile_count'
print(df[target].describe())
print(f"\nZero count: {(df[target] == 0).sum()} ({(df[target] == 0).mean()*100:.1f}%)")
print(f"Non-zero median: {df.loc[df[target] > 0, target].median():.1f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram (log scale)
axes[0].hist(df[target][df[target] > 0], bins=100, color='steelblue', edgecolor='white')
axes[0].set_xlabel('tile_count (>0)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of tile_count (excl. zeros)')

# Box plot
axes[1].boxplot(df[target][df[target] > 0], vert=False)
axes[1].set_xlabel('tile_count (>0)')
axes[1].set_title('Box Plot (excl. zeros)')

# CDF
sorted_vals = np.sort(df[target][df[target] > 0])
cdf = np.arange(1, len(sorted_vals)+1) / len(sorted_vals)
axes[2].plot(sorted_vals, cdf, color='steelblue')
axes[2].set_xlabel('tile_count')
axes[2].set_ylabel('CDF')
axes[2].set_title('Cumulative Distribution')
axes[2].set_xlim(0, 200)

plt.tight_layout()
plt.savefig('../outputs/figures/01_tile_count_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## Prefix Grouping

In [ ]:
from collections import Counter

prefixes = Counter(get_prefix(c) for c in df.columns)
prefix_df = pd.DataFrame(prefixes.most_common(), columns=['prefix', 'count'])
print(prefix_df.head(20))

In [ ]:
# Show columns by prefix for key groups
key_prefixes = ['solar', 'electricity', 'heating', 'cooling', 'education', 'population', 'housing', 'incentive', 'race', 'voting']
for p in key_prefixes:
    cols = [c for c in df.columns if get_prefix(c) == p]
    print(f"\n{p} ({len(cols)}): {cols}")

## Missing Values

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print(f"Columns with missing values: {len(missing)} / {len(df.columns)}")
print(missing.head(20))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
missing_top = missing.head(30)
ax.barh(range(len(missing_top)), missing_top.values, color='coral')
ax.set_yticks(range(len(missing_top)))
ax.set_yticklabels(missing_top.index, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Missing Count')
ax.set_title('Top 30 Columns by Missing Values')
plt.tight_layout()
plt.savefig('../outputs/figures/01_missing_values.png', dpi=300, bbox_inches='tight')
plt.show()

## US-only vs Generalizable Features

In [ ]:
us_only = get_us_only_cols()
targets = get_target_cols()
us_only_exist = [c for c in us_only if c in df.columns]
targets_exist = [c for c in targets if c in df.columns]

print(f"US-only columns found: {len(us_only_exist)}")
print(f"Target columns found: {len(targets_exist)}")

generalizable = [c for c in df.columns if c not in us_only_exist + targets_exist]
print(f"Generalizable feature columns: {len(generalizable)}")
print(f"Total: {len(df.columns)}")

## Correlation with Target (Preview)

In [ ]:
# Numeric columns only for preview
numeric_df = df.select_dtypes(include=[np.number])
corr_with_target = numeric_df.corr()[target].drop(target).sort_values(key=abs, ascending=False)
print('Top 15 positively correlated:')
print(corr_with_target.head(15))
print('\nTop 15 negatively correlated:')
print(corr_with_target.tail(15))

## Save Processed Metadata

In [ ]:
# Save column metadata for downstream notebooks
meta = {
    'us_only_cols': us_only_exist,
    'target_cols': targets_exist,
    'generalizable_cols': generalizable,
    'missing_cols': missing.index.tolist(),
    'prefix_counts': dict(prefixes),
}
import json
with open('../data/processed/column_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Saved to data/processed/column_metadata.json')